# RAG Fundamentals

**Module:** 04 — RAG

Retrieval-Augmented Generation grounds LLMs in your corpus. Build the mental model from definition through a runnable core loop.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define RAG and contrast it with fine-tuning and long-context stuffing
- Explain why RAG improves freshness, privacy handling, and citations
- Trace the index-time vs query-time core loop
- Implement a minimal retrieve → pack → generate toy pipeline
- Separate retrieval failures from generation failures when debugging


## What is RAG?

**Definition.** **Retrieval-Augmented Generation (RAG)** retrieves external evidence at query time and conditions the LLM on that evidence instead of relying only on parametric memory in the weights.

**Why it matters.** Weights go stale, miss private docs, and hallucinate fluently. RAG updates knowledge via re-indexing and supports attributable answers.

**How it works.** Index time: clean → chunk → embed → store. Query time: process query → retrieve → (rerank) → pack context → generate → cite/validate.

**Intuition.** The LLM is a skilled intern with amnesia; RAG opens the right filing cabinet before they speak.

**Common pitfalls.**
- Calling any web-search chatbot 'RAG' without a controlled corpus
- Expecting retrieval to fix a bad prompt alone
- Never refreshing the index when sources change
- Treating top-k similarity as factual truth

**When to use.** Use when answers must reflect your changing/private corpus with citations.

### Key terms

- **corpus**: Authorized document collection
- **chunk**: Retrieval unit with text + metadata
- **grounding**: Keeping generation faithful to evidence

```mermaid
flowchart TB
  subgraph index [Index time]
    D[Documents] --> C[Chunk + enrich]
    C --> E[Embed]
    E --> V[(Hybrid index)]
  end
  subgraph query [Query time]
    Q[Query] --> R[Retrieve]
    V --> R
    R --> P[Pack prompt]
    P --> L[LLM]
    L --> A[Answer + citations]
  end
```

### RAG vs alternatives

| Approach | Freshness | Private data | Update cost | Best for |
|----------|-----------|--------------|-------------|----------|
| Prompt stuffing | Session | Yes | N/A | Tiny corpora |
| Fine-tuning | Slow | Possible | High | Style/skills |
| Long context | Per request | Yes | Re-send tokens | Few large docs |
| **RAG** | Re-index | Yes | Medium | Large KBs |


In [ ]:
# Demo 1 — RAG dataflow contract
from dataclasses import dataclass, field
from typing import Any

@dataclass
class Chunk:
    id: str
    text: str
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass
class RagResult:
    answer: str
    citations: list[str]
    scores: list[float]

print("index:", ["documents", "chunker", "embedder", "index"])
print("query:", ["query", "retriever", "packer", "llm"])
print("Chunk fields:", list(Chunk.__dataclass_fields__))


In [ ]:
# Demo 2 — parametric memory cannot see live facts
LIVE_FACT = "Refund window is 60 days"  # wiki updated last week
print("model cutoff example: 2023-10")
print("live fact must be retrieved:", LIVE_FACT)


In [ ]:
# Demo 3 — chat API shape with grounded context (placeholder key)
import json
YOUR_API_KEY = "YOUR_API_KEY"  # os.getenv("OPENAI_API_KEY")

def build_rag_request(query: str, contexts: list[str]) -> dict:
    ctx = "\n---\n".join(contexts)
    return {
        "model": "gpt-4.1-mini",
        "temperature": 0.2,
        "messages": [
            {"role": "system", "content": "Answer ONLY from CONTEXT. Cite [C#]. If unknown, say so."},
            {"role": "user", "content": f"CONTEXT:\n{ctx}\n\nQUESTION:\n{query}"},
        ],
    }

print(json.dumps(build_rag_request(
    "What is the refund window?",
    ["[C1] Refunds within 60 days of purchase."],
), indent=2)[:700])
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


### Try it yourself — What is RAG?

1. Define RAG in one paragraph for a non-technical stakeholder.
2. List three corpuses in your org that must not rely on model memory.
3. Draw index-time vs query-time boxes for a FAQ bot.


## Why RAG?

**Definition.** RAG is the default when you need **grounded**, **updatable**, and **attributable** answers over a corpus too large or private for naive prompting.

**Why it matters.** It reduces hallucination risk on enterprise facts, enables compliance citations, and separates knowledge ops from model ops.

**How it works.** Treat retrieval error as first-class: measure recall@k, rerank, filter by metadata, and refuse when evidence is weak.

**Intuition.** Fine-tuning teaches *how to speak*; RAG supplies *what is true today*.

**Common pitfalls.**
- Using RAG for pure style transfer
- No refusal path when retrieval is weak
- Evaluating only final prose while retrieval is broken
- Ignoring chunking quality

**When to use.** Support deflection, policy QA, internal search, research copilots.

### Decision cues

| Signal | Prefer |
|--------|--------|
| Facts change weekly | RAG |
| Need citations | RAG |
| New skill/format | Fine-tune / prompt |
| Tiny corpus fits cheaply | Long context |
| Exact IDs/SKUs | Hybrid lexical + vector |


In [ ]:
# Demo 1 — grounded vs ungrounded answers
def ungrounded(q): return f"Definitely 42 days for: {q}"
def grounded(q, evidence):
    return "I lack evidence." if not evidence else f"Based on policy: {evidence}"
q = "refund window?"
print(ungrounded(q))
print(grounded(q, None))
print(grounded(q, "Refunds within 60 days."))


In [ ]:
# Demo 2 — cost sketch stuff-all vs top-k
N, K, TOK = 5000, 6, 400
stuff, rag = N * TOK, K * TOK
print(f"stuff-all={stuff:,} tok  rag={rag:,} tok  ratio={rag/stuff:.4%}")


In [ ]:
# Demo 3 — citation audit
def audit(chunk_ids, store):
    missing = [c for c in chunk_ids if c not in store]
    if not chunk_ids: return ["no citations"]
    return missing or ["ok"]
store = {"C12": "Refunds within 60 days"}
print(audit(["C12"], store), audit([], store), audit(["C99"], store))


### Try it yourself — Why RAG?

1. Argue RAG vs fine-tune vs long context for one real use case.
2. Write a refusal policy for low-evidence queries.
3. Estimate monthly tokens for stuff-all vs top-6 on your corpus size.


## Core Loop

**Definition.** The **core loop** is ingest→chunk→embed→index, then query→retrieve→(rerank)→pack→generate→(cite/validate).

**Why it matters.** Every production bug maps to a stage; localize before rewriting the stack.

**How it works.** Instrument chunk counts, recall@k, latency per stage, groundedness, and cost. Keep embedder identity in index metadata.

**Intuition.** Index time catalogs the library; query time pulls shelves and footnotes.

**Common pitfalls.**
- No gold eval set
- Logging answers without retrieved chunks
- Changing embedders without re-indexing
- Packing near-duplicate chunks

**When to use.** Ship this thin vertical slice before agents/graphs/multi-hop.

```mermaid
flowchart LR
  Q[Query] --> E[Embed]
  E --> S[Search]
  S --> R[Rerank]
  R --> P[Pack]
  P --> G[Generate]
  G --> V[Validate]
```

```text
+---------------- RAG CORE LOOP ------------------+
| INDEX: docs -> clean -> chunk -> embed -> store |
| QUERY: ask  -> retrieve -> pack -> LLM -> cite  |
+-------------------------------------------------+
```


In [ ]:
# Demo 1 — toy numpy RAG loop
import numpy as np
from dataclasses import dataclass

@dataclass
class Chunk:
    id: str
    text: str

def embed(text, vocab):
    t = text.lower().split()
    v = np.array([t.count(w) for w in vocab], float)
    return v / (np.linalg.norm(v) + 1e-9)

docs = [
    Chunk("C1", "refunds are accepted within sixty days of purchase"),
    Chunk("C2", "shipping takes three to five business days"),
    Chunk("C3", "password resets live in account settings"),
]
vocab = sorted({w for d in docs for w in d.text.split()})
M = np.stack([embed(d.text, vocab) for d in docs])

def retrieve(q, k=2):
    s = M @ embed(q, vocab)
    idx = np.argsort(-s)[:k]
    return [(docs[i], float(s[i])) for i in idx]

hits = retrieve("how long for refunds?")
print(" | ".join(f"{c.id}:{sc:.2f}" for c, sc in hits))
print("Evidence:", " || ".join(c.text for c, _ in hits))


In [ ]:
# Demo 2 — stage timing instrumentation
import time
from contextlib import contextmanager

@contextmanager
def stage(name, bucket):
    t0 = time.perf_counter(); yield
    bucket[name] = (time.perf_counter() - t0) * 1000

t = {}
for name, delay in [("embed", 0.01), ("search", 0.02), ("pack", 0.005), ("llm", 0.05)]:
    with stage(name, t):
        time.sleep(delay)
for k, v in t.items():
    print(f"{k:8s} {v:6.1f} ms")
print(f"{'total':8s} {sum(t.values()):6.1f} ms")


In [ ]:
# Demo 3 — localize retrieval vs generation failure
def diagnose(retrieved_ok, faithful):
    return {
        (0, 0): "Fix retrieval first, then grounding",
        (0, 1): "Careful refusal — improve recall@k",
        (1, 0): "Retrieval OK — tighten prompt/temperature",
        (1, 1): "Healthy — watch eval regressions",
    }[(retrieved_ok, faithful)]

for a in (0, 1):
    for b in (0, 1):
        print(a, b, "->", diagnose(a, b))


In [ ]:
# Demo 4 — embedder compatibility guard
INDEX = {"embed_model": "text-embedding-3-small", "dim": 1536}
QUERY = {"embed_model": "text-embedding-3-large", "dim": 3072}

def compatible(i, q):
    if i["embed_model"] != q["embed_model"] or i["dim"] != q["dim"]:
        raise ValueError("re-index required")
    return "ok"

try:
    print(compatible(INDEX, QUERY))
except ValueError as e:
    print("blocked:", e)
print(compatible(INDEX, INDEX))


### Try it yourself — Core Loop

1. Add two more chunks and re-run retrieve for a paraphrase query.
2. Return citations from chunk ids alongside the evidence string.
3. Force an embedder mismatch and confirm the guard fires.


## Glossary

- **RAG**: Retrieve then generate conditioned on evidence
- **Top-k**: How many chunks enter packing/reranking
- **Hybrid search**: Lexical + dense retrieval combined


### Workshop drill — RAG Fundamentals (1)

Diagram the data flow on paper, then implement one missing log line per stage.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 1 — RAG Fundamentals
stages = ['ingest','chunk','embed','retrieve','pack','generate']
for s in stages:
    print(f'log.{{s}}.ok = ?')


### Workshop drill — RAG Fundamentals (2)

Create two adversarial queries (one ID-heavy, one paraphrase-heavy) and compare hits.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 2 — RAG Fundamentals
queries = ['error code E42-refund', 'how do I get my money back?']
for q in queries:
    print('Q:', q)
    print('  TODO: print top-3 ids')


### Workshop drill — RAG Fundamentals (3)

Write a refusal test: empty hits must not produce a confident numeric answer.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 3 — RAG Fundamentals
def must_refuse(hits):
    return (not hits) or hits[0].get('score',0) < 0.2
assert must_refuse([])
assert must_refuse([{'score': 0.05}])
assert not must_refuse([{'score': 0.9}])
print('refusal tests ok')


### Workshop drill — RAG Fundamentals (4)

Estimate cost: vary top_k and context tokens; print a small table.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 4 — RAG Fundamentals
rows = []
for k in [2,4,8,16]:
    toks = k*400
    rows.append((k, toks, round(toks/1e6*0.5, 5)))
print('k  ctx_toks  approx_$')
for r in rows:
    print(*r)


### Workshop drill — RAG Fundamentals (5)

Add one metadata field and filter it in retrieval (tenant or product).

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 5 — RAG Fundamentals
docs = [{'id':'a','tenant':'acme'},{'id':'b','tenant':'beta'}]
tenant='acme'
print([d for d in docs if d['tenant']==tenant])


## Summary & Key Takeaways

- RAG grounds answers in evidence you control and can refresh
- Index-time quality usually dominates query-time cleverness
- Debug retrieval and generation as separate failure modes
- Citations and refusals are core product behavior

### Practice

Build a 10-chunk corpus; query with paraphrase, keyword, and OOD prompts.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
